# Qwen3-style LLM from scratch, trained with Muon

A 32.2M-parameter decoder-only transformer — grouped-query attention with QK-Norm, RoPE, SwiGLU, RMSNorm pre-norm, tied embeddings — pretrained on cosmopedia-v2 with the Muon optimizer on a single T4.

Architecture follows the [Qwen3 Technical Report](https://arxiv.org/abs/2505.09388); the optimizer follows [Muon](https://kellerjordan.github.io/posts/muon/).

Run order: imports -> config -> model -> data -> training. See README.md for results and FIXES.md for the engineering log.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler

import math
import os
import pickle
import random
import time

import numpy as np
from datasets import load_dataset
from tqdm import tqdm
from transformers import AutoTokenizer

from dataclasses import dataclass
from typing import List, Optional

In [ ]:
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # deterministic and benchmark are mutually exclusive; reproducibility wins here
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f"Set all seeds to {seed}")

In [ ]:
@dataclass
class ModelConfig:
    # architecture
    d_model: int = 384
    n_heads: int = 8
    n_kv_heads: int = 4
    n_layers: int = 6
    d_ff: int = 1536
    max_seq_len: int = 512
    attention_bias: bool = False
    rms_norm_eps: float = 1e-6
    dropout: float = 0.1

    # optimization
    batch_size: int = 24
    gradient_accumulation_steps: int = 4
    max_steps: int = 500
    muon_lr: float = 0.01
    weight_decay: float = 0.1
    grad_clip: float = 1.0
    use_amp: bool = True

    # data
    num_documents: int = 2000
    max_tokens: int = 500_000

    # evaluation
    eval_every: int = 100
    eval_steps: int = 100

    vocab_size: Optional[int] = None

    def __post_init__(self):
        assert self.d_model % self.n_heads == 0, "d_model must be divisible by n_heads"
        assert self.n_heads % self.n_kv_heads == 0, "n_heads must be divisible by n_kv_heads"
        self.d_k = self.d_model // self.n_heads
        # how many query heads share one key/value head (GQA)
        self.n_kv_groups = self.n_heads // self.n_kv_heads

In [ ]:
def repeat_kv(hidden_states: torch.Tensor, n_rep: int) -> torch.Tensor:
    """(B, n_kv_heads, T, d_k) -> (B, n_kv_heads * n_rep, T, d_k) for grouped-query attention."""
    if n_rep == 1:
        return hidden_states

    batch, num_key_value_heads, slen, head_dim = hidden_states.shape
    hidden_states = hidden_states[:, :, None, :, :].expand(batch, num_key_value_heads, n_rep, slen, head_dim)
    return hidden_states.reshape(batch, num_key_value_heads * n_rep, slen, head_dim)

Using Muon Optimizer

Gradient descent says: "Change the weights in this direction, and also stretch/shrink them by this amount."

Muon says: "Let's keep the useful direction of the update while removing much of the unnecessary stretching/shrinking."

In [ ]:
def zeropower_via_newtonschulz5(G: torch.Tensor, steps: int = 5) -> torch.Tensor:
    """Quintic Newton-Schulz iteration: approximates the orthogonal factor U @ V.T of G."""
    assert G.ndim >= 2
    a, b, c = (3.4445, -4.7750, 2.0315)

    # float32 keeps the iteration stable on GPUs without native bfloat16 (e.g. T4)
    X = G.float()
    transposed = G.size(-2) > G.size(-1)
    if transposed:
        X = X.mT

    X = X / (X.norm(dim=(-2, -1), keepdim=True) + 1e-7)

    for _ in range(steps):
        A = X @ X.mT
        B = b * A + c * A @ A
        X = a * X + B @ X

    return X.mT if transposed else X


class Muon(torch.optim.Optimizer):
    """Momentum SGD whose update is orthogonalized before being applied. 2D parameters only."""

    def __init__(self, params, lr=0.02, momentum=0.95, nesterov=True, ns_steps=5):
        super().__init__(params, dict(lr=lr, momentum=momentum, nesterov=nesterov, ns_steps=ns_steps))

    @torch.no_grad()
    def step(self):
        for group in self.param_groups:
            momentum = group["momentum"]

            for p in group["params"]:
                if p.grad is None:
                    continue

                state = self.state[p]
                if "momentum_buffer" not in state:
                    state["momentum_buffer"] = torch.zeros_like(p.grad)

                buf = state["momentum_buffer"]
                buf.lerp_(p.grad, 1 - momentum)
                update = p.grad.lerp(buf, momentum) if group["nesterov"] else buf

                update = zeropower_via_newtonschulz5(update, group["ns_steps"])
                # rescale so the update norm is comparable across differently shaped matrices
                scale = max(1.0, p.size(-2) / p.size(-1)) ** 0.5
                p.add_(update.to(p.dtype), alpha=-group["lr"] * scale)

In [ ]:
def load_and_cache_data(config: ModelConfig, cache_dir: str = "./cache"):
    """Stream documents from cosmopedia-v2, tokenize them into one flat stream, and cache the result."""
    os.makedirs(cache_dir, exist_ok=True)
    cache_file = os.path.join(cache_dir, f"tokenized_data_{config.num_documents}_{config.max_tokens}.pkl")

    if os.path.exists(cache_file):
        with open(cache_file, "rb") as f:
            cached = pickle.load(f)
        config.vocab_size = cached["tokenizer"].vocab_size
        print(f"Loaded cache: {len(cached['tokens']):,} tokens, vocab {config.vocab_size:,}")
        return cached["texts"], cached["tokenizer"], cached["tokens"]

    tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM-135M")
    dataset = load_dataset("HuggingFaceTB/smollm-corpus", "cosmopedia-v2", split="train", streaming=True)

    texts = [item["text"][:3000] for _, item in zip(range(config.num_documents), dataset)]

    tokens: List[int] = []
    for text in tqdm(texts, desc="Tokenizing"):
        tokens.extend(tokenizer.encode(text, add_special_tokens=False))
        if len(tokens) >= config.max_tokens:
            break
    tokens = tokens[: config.max_tokens]

    config.vocab_size = tokenizer.vocab_size
    with open(cache_file, "wb") as f:
        pickle.dump({"texts": texts, "tokenizer": tokenizer, "tokens": tokens}, f)

    print(f"Tokenized {len(texts)} documents -> {len(tokens):,} tokens, vocab {config.vocab_size:,}")
    return texts, tokenizer, tokens

In [ ]:
class TextTokenDataset(Dataset):
    def __init__(self, tokens: List[int], seq_len: int = 512):
        self.tokens = tokens
        self.seq_len = seq_len

    def __len__(self):
        return max(0, len(self.tokens) - self.seq_len)

    def __getitem__(self, idx):
        x = torch.tensor(self.tokens[idx:idx + self.seq_len], dtype=torch.long)
        y = torch.tensor(self.tokens[idx + 1:idx + self.seq_len + 1], dtype=torch.long)
        return x, y

In [ ]:
class Rotary(nn.Module):
    """Rotary position embeddings applied to the first half of each head's channels."""

    def __init__(self, dim: int, max_seq_len: int):
        super().__init__()
        angular_freq = (1 / 10000) ** torch.linspace(0, 1, steps=dim // 4, dtype=torch.float32)
        angular_freq = torch.cat([angular_freq, angular_freq.new_zeros(dim // 4)])
        theta = torch.outer(torch.arange(max_seq_len, dtype=torch.float32), angular_freq)
        self.register_buffer("cos", theta.cos(), persistent=False)
        self.register_buffer("sin", theta.sin(), persistent=False)

    def forward(self, x_BTHD: torch.Tensor):
        seq_len = x_BTHD.size(-3)
        cos = self.cos[None, :seq_len, None, :]
        sin = self.sin[None, :seq_len, None, :]

        x1, x2 = x_BTHD.float().chunk(2, dim=-1)
        y1 = x1 * cos + x2 * sin
        y2 = x1 * (-sin) + x2 * cos
        return torch.cat((y1, y2), dim=-1).type_as(x_BTHD)

In [ ]:
class Qwen3Attention(nn.Module):
    def __init__(self, config: ModelConfig):
        super().__init__()
        self.n_heads = config.n_heads
        self.n_kv_heads = config.n_kv_heads
        self.n_kv_groups = config.n_kv_groups
        self.d_k = config.d_k

        self.q_proj = nn.Linear(config.d_model, self.n_heads * self.d_k, bias=config.attention_bias)
        self.k_proj = nn.Linear(config.d_model, self.n_kv_heads * self.d_k, bias=config.attention_bias)
        self.v_proj = nn.Linear(config.d_model, self.n_kv_heads * self.d_k, bias=config.attention_bias)
        self.o_proj = nn.Linear(self.n_heads * self.d_k, config.d_model, bias=False)

        # Qwen3-style per-head QK normalization
        self.q_norm = nn.RMSNorm(self.d_k, eps=config.rms_norm_eps)
        self.k_norm = nn.RMSNorm(self.d_k, eps=config.rms_norm_eps)

        self.rotary = Rotary(self.d_k, config.max_seq_len)
        self.dropout = config.dropout

    def forward(self, x):
        batch_size, seq_len, _ = x.shape

        q = self.q_proj(x).view(batch_size, seq_len, self.n_heads, self.d_k)
        k = self.k_proj(x).view(batch_size, seq_len, self.n_kv_heads, self.d_k)
        v = self.v_proj(x).view(batch_size, seq_len, self.n_kv_heads, self.d_k)

        # Rotary expects (B, T, H, D); move to (B, H, T, D) only afterwards for attention
        q = self.rotary(self.q_norm(q)).transpose(1, 2)
        k = self.rotary(self.k_norm(k)).transpose(1, 2)
        v = v.transpose(1, 2)

        k = repeat_kv(k, self.n_kv_groups)
        v = repeat_kv(v, self.n_kv_groups)

        attn_output = F.scaled_dot_product_attention(
            q, k, v, is_causal=True, dropout_p=self.dropout if self.training else 0.0
        )

        attn_output = attn_output.transpose(1, 2).reshape(batch_size, seq_len, -1)
        return self.o_proj(attn_output)

In [ ]:
class SwiGLUFeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.gate_proj = nn.Linear(d_model, d_ff, bias=False)
        self.down_proj = nn.Linear(d_ff, d_model, bias=False)
        self.up_proj = nn.Linear(d_model, d_ff, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        activated_x = F.silu(self.gate_proj(x)) * self.up_proj(x)
        return self.down_proj(self.dropout(activated_x))


In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, config: ModelConfig):
        super().__init__()
        self.attention = Qwen3Attention(config)
        self.feed_forward = SwiGLUFeedForward(config.d_model, config.d_ff, config.dropout)
        self.norm1 = nn.RMSNorm(config.d_model, eps=config.rms_norm_eps)
        self.norm2 = nn.RMSNorm(config.d_model, eps=config.rms_norm_eps)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        attn_out = self.attention(self.norm1(x))
        x = x + self.dropout(attn_out)
        ff_out = self.feed_forward(self.norm2(x))
        x = x + self.dropout(ff_out)
        return x


In [ ]:
class MinimalLLM(nn.Module):
    def __init__(self, config: ModelConfig):
        super().__init__()
        self.config = config

        # positions come from RoPE inside attention, so there is no learned position embedding
        self.token_embedding = nn.Embedding(config.vocab_size, config.d_model)
        self.input_dropout = nn.Dropout(config.dropout)
        self.transformer_blocks = nn.ModuleList([TransformerBlock(config) for _ in range(config.n_layers)])
        self.norm = nn.RMSNorm(config.d_model, eps=config.rms_norm_eps)
        self.output_dropout = nn.Dropout(config.dropout)

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
        if isinstance(module, nn.Linear) and module.bias is not None:
            nn.init.zeros_(module.bias)

    def forward(self, x):
        x = self.input_dropout(self.token_embedding(x))

        for block in self.transformer_blocks:
            x = block(x)

        x = self.output_dropout(self.norm(x))
        # weight tying: the LM head reuses the embedding matrix
        return F.linear(x, self.token_embedding.weight)

In [ ]:
@torch.no_grad()
def evaluate_model(model: nn.Module, val_loader: DataLoader, config: ModelConfig):
    model.eval()
    device = next(model.parameters()).device

    total_loss, total_tokens, total_correct = 0.0, 0, 0

    for i, (x, y) in enumerate(val_loader):
        if i >= config.eval_steps:
            break

        x, y = x.to(device), y.to(device)

        with autocast(device.type, enabled=config.use_amp):
            logits = model(x)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), y.view(-1))

        total_loss += loss.item() * y.numel()
        total_tokens += y.numel()
        total_correct += (logits.argmax(dim=-1) == y).sum().item()

    model.train()

    avg_loss = total_loss / total_tokens
    return {
        'val_loss': avg_loss,
        'val_accuracy': total_correct / total_tokens,
        'val_perplexity': math.exp(min(avg_loss, 20)),
    }

In [ ]:
def setup_muon_optimizer(model: nn.Module, config: ModelConfig):
    """Muon for the 2D hidden matrices, AdamW for embeddings, norms and biases."""
    muon_params, adamw_params = [], []

    for name, param in model.named_parameters():
        if param.ndim == 2 and 'token_embedding' not in name:
            muon_params.append(param)
        else:
            adamw_params.append(param)

    print(f"  Muon parameters: {sum(p.numel() for p in muon_params):,}")
    print(f"  AdamW parameters: {sum(p.numel() for p in adamw_params):,}")

    muon_optimizer = Muon(muon_params, lr=config.muon_lr, momentum=0.95)
    adamw_optimizer = torch.optim.AdamW(
        adamw_params, lr=config.muon_lr * 0.1, weight_decay=config.weight_decay
    )

    return [muon_optimizer, adamw_optimizer]

In [ ]:
def train_model(config: ModelConfig, train_loader: DataLoader, val_loader: DataLoader):
    """Train the model with Muon (+ AdamW). One `step` is one optimizer update."""
    set_seed(42)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = MinimalLLM(config).to(device)
    print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")

    optimizers = setup_muon_optimizer(model, config)

    warmup_steps = max(1, config.max_steps // 20)

    def lr_lambda(step):
        if step < warmup_steps:
            return (step + 1) / warmup_steps
        progress = (step - warmup_steps) / max(1, config.max_steps - warmup_steps)
        return 0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * progress))

    schedulers = [torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda) for opt in optimizers]

    # when use_amp is False every scaler call becomes a no-op, so there is only one code path
    scaler = GradScaler(device.type, enabled=config.use_amp)

    def endless(loader):
        while True:
            yield from loader

    batches = endless(train_loader)

    model.train()
    best_val_loss = float('inf')
    start_time = time.time()

    pbar = tqdm(range(config.max_steps), desc="Training")
    for step in pbar:
        for _ in range(config.gradient_accumulation_steps):
            x, y = next(batches)
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)

            with autocast(device.type, enabled=config.use_amp):
                logits = model(x)
                loss = F.cross_entropy(logits.view(-1, logits.size(-1)), y.view(-1))

            scaler.scale(loss / config.gradient_accumulation_steps).backward()

        for optimizer in optimizers:
            scaler.unscale_(optimizer)

        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), config.grad_clip)

        for optimizer in optimizers:
            scaler.step(optimizer)
            optimizer.zero_grad(set_to_none=True)

        scaler.update()

        for scheduler in schedulers:
            scheduler.step()

        if step % 10 == 0:
            with torch.no_grad():
                accuracy = (logits.argmax(dim=-1) == y).float().mean().item()
            current_loss = loss.item()

            pbar.set_postfix({
                'loss': f'{current_loss:.4f}',
                'acc': f'{accuracy:.3f}',
                'ppl': f'{math.exp(min(current_loss, 20)):.1f}',
                'gnorm': f'{grad_norm:.2f}',
                'lr': f'{optimizers[0].param_groups[0]["lr"]:.2e}',
            })

        if step > 0 and step % config.eval_every == 0:
            eval_metrics = evaluate_model(model, val_loader, config)

            print(
                f"Step {step}: "
                f"Val Loss: {eval_metrics['val_loss']:.4f}, "
                f"Val Acc: {eval_metrics['val_accuracy']:.4f}, "
                f"Val PPL: {eval_metrics['val_perplexity']:.2f}"
            )

            if eval_metrics['val_loss'] < best_val_loss:
                best_val_loss = eval_metrics['val_loss']

                torch.save({
                    'model_state_dict': model.state_dict(),
                    'config': config,
                    'step': step,
                    'best_val_loss': best_val_loss,
                    'final_metrics': eval_metrics,
                }, 'best_model.pt')

    pbar.close()

    final_eval = evaluate_model(model, val_loader, config)

    print(
        f"Training time: {(time.time() - start_time) / 60:.1f} min | "
        f"Final - Loss: {final_eval['val_loss']:.4f}, "
        f"Acc: {final_eval['val_accuracy']:.4f}, "
        f"PPL: {final_eval['val_perplexity']:.2f}"
    )

    torch.save({
        'model_state_dict': model.state_dict(),
        'config': config,
        'step': config.max_steps,
        'final_metrics': final_eval,
    }, 'final_model.pt')

    return model, final_eval

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(
        f"Memory: "
        f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB"
    )

set_seed(42)

config = ModelConfig()

print("Model Configuration:")
print(
    f"Architecture: {config.d_model}d, "
    f"{config.n_layers}L, "
    f"{config.n_heads}H ({config.n_kv_heads} KV), "
    f"{config.d_ff}ff"
)
print(
    f"Training: {config.max_steps} steps, "
    f"batch size {config.batch_size} x {config.gradient_accumulation_steps} accum"
)
print(
    f"Data: {config.max_tokens:,} tokens, "
    f"seq_len {config.max_seq_len}"
)

texts, tokenizer, tokens = load_and_cache_data(config, "./cache")

dataset = TextTokenDataset(tokens, config.max_seq_len)

val_size = len(dataset) // 10
train_size = len(dataset) - val_size

train_dataset, val_dataset = torch.utils.data.random_split(
    dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(
    train_dataset,
    batch_size=config.batch_size,
    shuffle=True,
    drop_last=True,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset,
    batch_size=config.batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

print(
    f"Dataset: {len(train_dataset)} train, "
    f"{len(val_dataset)} val samples"
)

model, final_metrics = train_model(config, train_loader, val_loader)

print(f"Validation Loss: {final_metrics['val_loss']:.4f}")
print(f"Validation Accuracy: {final_metrics['val_accuracy']:.4f}")
print(f"Validation Perplexity: {final_metrics['val_perplexity']:.2f}")

In [ ]:
def load_trained_model(model_path: str = "final_model.pt"):
    """Load a trained model from a checkpoint."""
    torch.serialization.add_safe_globals([ModelConfig])
    checkpoint = torch.load(model_path, map_location="cpu", weights_only=True)

    config = checkpoint["config"]
    model = MinimalLLM(config)
    model.load_state_dict(checkpoint["model_state_dict"])

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device).eval()

    print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
    print(f"Device: {device}")

    return model, config

In [ ]:
@torch.no_grad()
def generate_text(
    model: nn.Module,
    tokenizer,
    prompt: str,
    max_new_tokens: int = 100,
    temperature: float = 0.8,
    top_k: int = 50,
    top_p: float = 0.9
):
    """Generate text using the trained model."""
    model.eval()
    device = next(model.parameters()).device
    max_seq_len = model.config.max_seq_len

    generated_ids = tokenizer.encode(
        prompt,
        add_special_tokens=False,
        return_tensors="pt"
    ).to(device)

    for _ in range(max_new_tokens):
        # the model has no KV cache, so feed the last max_seq_len tokens of context
        logits = model(generated_ids[:, -max_seq_len:])
        next_token_logits = logits[0, -1] / temperature

        if top_k > 0:
            kth_value = torch.topk(
                next_token_logits,
                min(top_k, next_token_logits.size(-1))
            ).values[-1]
            next_token_logits[next_token_logits < kth_value] = float("-inf")

        if top_p < 1.0:
            sorted_logits, sorted_indices = torch.sort(
                next_token_logits,
                descending=True
            )

            cumulative_probs = F.softmax(sorted_logits, dim=-1).cumsum(dim=-1)

            # keep the first token that crosses top_p, drop everything after it
            sorted_indices_to_remove = cumulative_probs > top_p
            sorted_indices_to_remove[1:] = sorted_indices_to_remove[:-1].clone()
            sorted_indices_to_remove[0] = False

            next_token_logits[sorted_indices[sorted_indices_to_remove]] = float("-inf")

        probs = F.softmax(next_token_logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)

        generated_ids = torch.cat([generated_ids, next_token.view(1, 1)], dim=1)

        if next_token.item() == tokenizer.eos_token_id:
            break

    return tokenizer.decode(generated_ids[0], skip_special_tokens=True)